# 3D Gaussian Splatting BTS Digital Twin Pipeline

Notebook này hướng dẫn bạn chạy thử nghiệm (dry-run) và chạy huấn luyện chính thức (production) thuật toán 3DGS tái dựng trạm BTS trên Google Colab GPU.

## Bước 1: Chuẩn bị môi trường & Giải nén Code
Nếu bạn nén toàn bộ thư mục code này thành file `code.zip` và upload lên Colab, hãy chạy cell dưới đây để giải nén.

In [ ]:
# Giải nén mã nguồn (thay đổi tên file nếu cần)
# !unzip code.zip -d ./3dgs_bts
# %cd ./3dgs_bts

## Bước 2: Cài đặt thư viện và Biên dịch CUDA Submodules
Google Colab cung cấp đầy đủ CUDA Compiler (nvcc) và C++ build tools. Việc cài đặt các package custom rasterization sẽ thành công 100%.

In [ ]:
# Cài đặt các thư viện Python bổ trợ
!pip install pandas opencv-python joblib plyfile

# Cài đặt và biên dịch các module CUDA của 3DGS
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn
!pip install submodules/fused-ssim

## Bước 3: Chạy Thử Nghiệm Local (Dry-Run Mode)
Chúng ta sẽ tạo dữ liệu giả lập có kích thước nhỏ (64x64) và chạy huấn luyện 100 iterations để kiểm tra tính toàn vẹn của pipeline, xuất ảnh test poses, kiểm định kích thước và đóng gói zip.

In [ ]:
# 1. Sinh dữ liệu giả lập
!python create_mock_data.py

# 2. Chạy thử nghiệm pipeline chế độ test
!python run_pipeline.py --mode test --data_root ./mock_data --output_dir ./mock_submit --iterations 100

Nếu bước chạy trên báo `All checks passed! Packaging submission.zip...` và sinh ra file `mock_submit/submission.zip` thành công, pipeline đã hoàn toàn hoạt động ổn định và sẵn sàng cho dữ liệu thật!

## Bước 4: Tải Trọng Số Cho Depth Anything v2 (Chạy huấn luyện thật)
Để đạt chất lượng hình ảnh cao nhất cho cấu trúc trạm BTS mảnh, chúng ta cần sinh bản đồ độ sâu trước khi train.

In [ ]:
# 1. Clone repository Depth Anything v2 nếu chưa có
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

# 2. Tạo thư mục checkpoints và tải trọng số Vit-Large
!mkdir -p Depth-Anything-V2/checkpoints
!wget -O Depth-Anything-V2/checkpoints/depth_anything_v2_vitl.pth https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth

## Bước 5: Chạy Huấn Luyện SOTA & Kết Xuất Nộp Bài
Nạp dataset thật của ban tổ chức vào (ví dụ đường dẫn `/content/drive/MyDrive/BTS_Dataset`), sau đó chạy huấn luyện chất lượng cao nhất (30k iterations, Depth Regularization, EWA Anti-aliasing, Histogram Color Matching, Sharpening).

In [ ]:
# Điền đường dẫn dataset thật của bạn vào đây
DATASET_REAL = "/path/to/real_dataset"
SUBMISSION_OUT = "./final_submit"

# Chạy pipeline chế độ Production
!python run_pipeline.py --mode prod --data_root {DATASET_REAL} --output_dir {SUBMISSION_OUT}